# 04 — AOI Vegetation-Index Time Series, Savitzky-Golay Smoothing & Client-Side Re-Filtering

**What:** Build an AOI-average vegetation-index (NDVI) time series from the Sentinel-2 SR Harmonized collection, plot it with Plotly, smooth it with a Savitzky-Golay filter (`scipy.signal.savgol_filter`), and demonstrate the **new client-side filtering architecture**.

**Why:** This is the optical core of the legacy RAVI plugin that the current fork is missing. The new architecture attaches per-image filter **metadata** (cloud %, valid-pixel %, AOI coverage %, date) to every time-series row so the Results panel "Adjust filter" popup can **re-filter the cached pandas DataFrame entirely client-side** — no new Earth Engine round-trips.

**Legacy reference** (`git show legacy:ravi_dialog.py`):
- `calculate_timeseries` (~L2461) + `calculate_index_with_mean` (~L2493): maps a `reduceRegion(ee.Reducer.mean())` per image, then `aggregate_array("date"/"mean_index"/"system:index").getInfo()` into a DataFrame.
- DataFrame columns (`df_ajust`, ~L1385): `date`, `AOI_average`, `savitzky_golay_filtered`, `image_id`.
- `df_run_filter` (~L3993): `savgol_filter(df["AOI_average"], window_length, polyorder)`.
- `plot_timeseries` (~L4007): two `go.Scatter` traces (raw green, filtered purple).
- `AOI_coverage_filter` (~L3742): coverage ratio = AOI∩footprint area / AOI area.
- `SCL_filter` (~L3808): valid-pixel % = masked SCL pixel count / total pixel count, both via `reduceRegion(ee.Reducer.count())` at scale 10.
- Tile cloud filter (~L3650): `CLOUDY_PIXEL_PERCENTAGE`.

> **Filtering-architecture note:** Legacy applied cloud/coverage/valid-pixel thresholds *server-side* (re-querying Earth Engine on every slider change — slow). The new design queries **once**, carries the metrics as columns, and lets the user re-tune thresholds against the cached DataFrame instantly.

In [ ]:
# --- Setup ---
import os
import ee
import pandas as pd
import plotly.graph_objects as go
from scipy.signal import savgol_filter

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)
print("Earth Engine initialised.")

In [ ]:
# --- AOI + date range + filter parameters ---
# AOI loaded from the project shapefile (same area as dev/testing.ipynb).
# Legacy keeps self.aoi as an ee.FeatureCollection, so the loader returns one.
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


aoi = load_aoi_from_shapefile("contorno_area_total.zip")

INICIO = "2023-01-01"   # legacy: self.inicio
FINAL = "2023-12-31"    # legacy: self.final
NUVEM = 80              # legacy: self.nuvem -> CLOUDY_PIXEL_PERCENTAGE upper bound (tile)
VEGETATION_INDEX = "NDVI"

print(f"AOI: {aoi.size().getInfo()} feature(s); {INICIO} .. {FINAL}; tile cloud < {NUVEM}%")

## Build the masked S2 collection + NDVI band

Mirrors legacy `QPushButton_dates_clicked` collection build: `COPERNICUS/S2_SR_HARMONIZED`, filtered by date / bounds / `CLOUDY_PIXEL_PERCENTAGE`, tagged with a `date` property. NDVI follows the legacy lambda `image.normalizedDifference(["B8","B4"]).rename("index")`.

In [ ]:
# --- S2 collection (legacy: build + tile cloud filter) ---
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(INICIO, FINAL)
    .filterBounds(aoi)
    .map(lambda image: image.set("date", image.date().format("YYYY-MM-dd")))
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", NUVEM))
)


def calculate_vegetation_index(image, index_name):
    """Legacy calculate_vegetation_index (subset). Returns a single 'index' band."""
    formulas = {
        "NDVI": lambda img: img.normalizedDifference(["B8", "B4"]).rename("index"),
        "GNDVI": lambda img: img.normalizedDifference(["B8", "B3"]).rename("index"),
        "NDRE": lambda img: img.normalizedDifference(["B8", "B5"]).rename("index"),
    }
    return formulas[index_name](image)


print("Collection size after tile cloud filter:", sentinel2.size().getInfo())

## Map a reducer over the collection -> per-date AOI mean + filter metadata -> pandas

Legacy `calculate_index_with_mean` only carried `mean_index`. The **new architecture** sets four extra metadata properties per image so they ride along into the cached DataFrame:

| Column | Source (legacy logic) |
|---|---|
| `AOI_average` | `index.reduceRegion(ee.Reducer.mean(), aoi, scale=10)` — legacy `calculate_index_with_mean` |
| `cloud_pct` | `CLOUDY_PIXEL_PERCENTAGE` tile property — legacy cloud filter |
| `valid_pixel_pct` | SCL valid count / total count * 100 — legacy `SCL_filter` |
| `coverage_pct` | AOI∩footprint area / AOI area * 100 — legacy `AOI_coverage_filter` |
| `time_start` | `system:time_start` |
| `image_id` | `system:index` — legacy `calculate_timeseries` |

Everything is computed server-side in **one** `aggregate_array(...).getInfo()` batch — afterwards we never call Earth Engine again.

In [ ]:
# --- Attach AOI mean + filter metadata to every image (server-side, one pass) ---
aoi_geom = aoi.first().geometry()
aoi_area = aoi_geom.area()

# SCL classes treated as INVALID (legacy default: cloud shadow, clouds, cirrus).
INVALID_SCL = [3, 8, 9, 10]


def annotate(image):
    """Per-image: AOI mean index + cloud/valid-pixel/coverage metadata."""
    index_image = calculate_vegetation_index(image, VEGETATION_INDEX)

    # AOI_average (legacy calculate_index_with_mean)
    mean_index = index_image.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi_geom, scale=10, bestEffort=True
    ).get("index")

    # Valid-pixel % (legacy SCL_filter): keep pixels NOT in INVALID_SCL.
    scl = image.select("SCL")
    valid_mask = scl.remap(INVALID_SCL, [0] * len(INVALID_SCL), 1)  # 0 if invalid, else 1
    total_pixels = (
        image.select(0)
        .reduceRegion(reducer=ee.Reducer.count(), geometry=aoi_geom, scale=10, bestEffort=True)
        .get("B1")
    )
    valid_pixels = (
        image.select(0).updateMask(valid_mask)
        .reduceRegion(reducer=ee.Reducer.count(), geometry=aoi_geom, scale=10, bestEffort=True)
        .get("B1")
    )
    valid_pixel_pct = ee.Number(valid_pixels).divide(ee.Number(total_pixels).max(1)).multiply(100)

    # AOI coverage % (legacy AOI_coverage_filter): intersection area / AOI area.
    intersection = aoi_geom.intersection(image.geometry(), ee.ErrorMargin(1))
    coverage_pct = intersection.area().divide(aoi_area).multiply(100)

    return image.set({
        "mean_index": mean_index,
        "valid_pixel_pct": valid_pixel_pct,
        "coverage_pct": coverage_pct,
    })


annotated = sentinel2.map(annotate).filter(ee.Filter.notNull(["mean_index"]))

# Single batched pull (legacy aggregate_array(...).getInfo() pattern).
dates = annotated.aggregate_array("date").getInfo()
mean_indices = annotated.aggregate_array("mean_index").getInfo()
image_ids = annotated.aggregate_array("system:index").getInfo()
cloud_pcts = annotated.aggregate_array("CLOUDY_PIXEL_PERCENTAGE").getInfo()
valid_pcts = annotated.aggregate_array("valid_pixel_pct").getInfo()
coverage_pcts = annotated.aggregate_array("coverage_pct").getInfo()
time_starts = annotated.aggregate_array("system:time_start").getInfo()

# CACHED DataFrame — the single source of truth for all subsequent filtering.
df_cache = pd.DataFrame({
    "date": pd.to_datetime(dates),
    "AOI_average": mean_indices,
    "cloud_pct": cloud_pcts,
    "valid_pixel_pct": valid_pcts,
    "coverage_pct": coverage_pcts,
    "time_start": time_starts,
    "image_id": image_ids,
}).sort_values("date").reset_index(drop=True)

print(f"Cached {len(df_cache)} dates. NO further Earth Engine calls below this line.")
df_cache.head()

## Plot the raw AOI-average series (Plotly)

Legacy `plot_timeseries`: raw `AOI_average` as a green line.

In [ ]:
fig_raw = go.Figure()
fig_raw.add_trace(go.Scatter(
    x=df_cache["date"], y=df_cache["AOI_average"],
    mode="lines+markers", name=VEGETATION_INDEX, line=dict(color="green"),
))
fig_raw.update_layout(
    title=f"Time Series - {VEGETATION_INDEX} - AOI average     Image count: {len(df_cache)}",
    yaxis_title=VEGETATION_INDEX, xaxis_title=None,
)
fig_raw.update_traces(
    hovertemplate="date = %{x|%Y-%m-%d}<br>average = %{y:.2f}<extra></extra>"
)
fig_raw.show()

## Savitzky-Golay smoothing

Legacy `df_run_filter`: `savgol_filter(df["AOI_average"], window_length, polyorder)`. `window_length` must be odd and `<= len(df)`; `polyorder < window_length`. The smoothed values become the `savitzky_golay_filtered` column and overlay as a purple line (legacy `plot_timeseries`).

In [ ]:
def apply_savgol(df, window_length=7, polyorder=2):
    """Legacy df_run_filter logic with guards for short series."""
    out = df.copy()
    n = len(out)
    wl = min(window_length, n)
    if wl % 2 == 0:  # savgol requires an odd window
        wl -= 1
    if wl < 3:
        raise ValueError("Not enough images to apply the Savitzky-Golay filter.")
    po = min(polyorder, wl - 1)
    out["savitzky_golay_filtered"] = savgol_filter(
        out["AOI_average"], window_length=wl, polyorder=po
    )
    print(f"Savitzky-Golay applied: window_length={wl}, polyorder={po}")
    return out


df_cache = apply_savgol(df_cache, window_length=7, polyorder=2)

fig_smooth = go.Figure()
fig_smooth.add_trace(go.Scatter(
    x=df_cache["date"], y=df_cache["AOI_average"],
    mode="lines", name=VEGETATION_INDEX, line=dict(color="green"),
))
fig_smooth.add_trace(go.Scatter(
    x=df_cache["date"], y=df_cache["savitzky_golay_filtered"],
    mode="lines", name=f"{VEGETATION_INDEX} filtered", line=dict(color="purple"),
))
fig_smooth.update_layout(
    title=f"Time Series - {VEGETATION_INDEX} (raw + Savitzky-Golay)",
    yaxis_title=VEGETATION_INDEX, xaxis_title=None,
)
fig_smooth.show()

## Client-side re-filter — the "Adjust filter" popup (NEW architecture)

The cached DataFrame already carries every metric the legacy plugin re-queried Earth Engine for. The Results-panel "Adjust filter" popup simply applies pandas boolean masks to `df_cache`:

- **max cloud %** -> `cloud_pct <= max_cloud_pct`
- **min valid-pixel %** -> `valid_pixel_pct >= min_valid_pixel_pct`
- **min coverage %** -> `coverage_pct >= min_coverage_pct`
- **date subset** -> `date.between(start, end)`

Savitzky-Golay is re-fit on the surviving rows. **No `ee.*` calls anywhere in this cell** — the user can drag sliders and re-plot with zero latency.

In [ ]:
def adjust_filter(
    df,
    max_cloud_pct=40,
    min_valid_pixel_pct=60,
    min_coverage_pct=90,
    date_start=None,
    date_end=None,
    savgol_window=7,
    savgol_polyorder=2,
):
    """Pure-pandas re-filter of the cached time series. NO Earth Engine calls."""
    mask = (
        (df["cloud_pct"] <= max_cloud_pct)
        & (df["valid_pixel_pct"] >= min_valid_pixel_pct)
        & (df["coverage_pct"] >= min_coverage_pct)
    )
    if date_start is not None:
        mask &= df["date"] >= pd.to_datetime(date_start)
    if date_end is not None:
        mask &= df["date"] <= pd.to_datetime(date_end)

    out = df.loc[mask, ["date", "AOI_average", "cloud_pct", "valid_pixel_pct",
                        "coverage_pct", "image_id"]].reset_index(drop=True)
    print(f"Re-filtered client-side: {len(out)}/{len(df)} dates kept (no GEE round-trip).")
    if len(out) >= 3:
        out = apply_savgol(out, window_length=savgol_window, polyorder=savgol_polyorder)
    return out


# Example: tighten cloud/coverage and zoom into the growing season.
df_filtered = adjust_filter(
    df_cache,
    max_cloud_pct=30,
    min_valid_pixel_pct=70,
    min_coverage_pct=95,
    date_start="2023-03-01",
    date_end="2023-09-30",
)
df_filtered.head()

In [ ]:
# --- Re-plot the client-side-filtered series ---
fig_filt = go.Figure()
fig_filt.add_trace(go.Scatter(
    x=df_cache["date"], y=df_cache["AOI_average"],
    mode="lines", name="all dates (cached)", line=dict(color="lightgray"),
))
fig_filt.add_trace(go.Scatter(
    x=df_filtered["date"], y=df_filtered["AOI_average"],
    mode="lines+markers", name=f"{VEGETATION_INDEX} (filtered)", line=dict(color="green"),
))
if "savitzky_golay_filtered" in df_filtered:
    fig_filt.add_trace(go.Scatter(
        x=df_filtered["date"], y=df_filtered["savitzky_golay_filtered"],
        mode="lines", name=f"{VEGETATION_INDEX} filtered (SavGol)", line=dict(color="purple"),
    ))
fig_filt.update_layout(
    title=f"Adjust filter (client-side) - {VEGETATION_INDEX}     Kept: {len(df_filtered)}/{len(df_cache)}",
    yaxis_title=VEGETATION_INDEX, xaxis_title=None,
)
fig_filt.show()

print("Drag the thresholds and re-run adjust_filter(...) — every re-plot is instant,")
print("because it reads df_cache only. The Earth Engine query happened exactly once.")